# Fairness Check


In [ ]:
# import required libraries and add project root to the path
import sys
sys.path.append("../")

from multiple_stage_notebooks import *

## 1. Load data, sensitive attributes, and the final model

**Industry rule:** fairness must always be checked against the exact same model and predictions that will actually go to production — never against a leftover experiment. Load everything from the single source of truth (`06_model_training.ipynb`'s saved outputs), not from ad-hoc variables left over in memory.

In [2]:
data = np.load("../artifacts/splits/x_y_splits.npz")
raw_X_train_prep = data['raw_X_train_prep']
raw_X_test_prep = data['raw_X_test_prep']
y_train = data['y_train']
y_test = data['y_test']

# Raw (unprocessed) frames — needed because sensitive attributes like age_group
# are human-readable columns, not the scaled/encoded numeric columns models train on.
scaled_raw_data_X_train = pd.read_csv("../data/processed/scaled_raw_data_X_train.csv")
scaled_raw_data_X_test = pd.read_csv("../data/processed/scaled_raw_data_X_test.csv")

raw_X_train = pd.read_csv("../data/processed/raw_X_train.csv")
raw_X_test = pd.read_csv("../data/processed/raw_X_test.csv")

# Rule: X_train/X_test row order MUST match scaled_data_X_train/scaled_data_X_test exactly,
# since they come from separate files. Always assert this — silent misalignment
# is a classic invisible bug that produces confidently wrong fairness numbers.
assert len(scaled_raw_data_X_train) == len(raw_X_train), "Row count mismatch: raw_X_train vs scaled_raw_data_X_train"
assert len(scaled_raw_data_X_test) == len(raw_X_test), "Row count mismatch: raw_X_test vs scaled_raw_data_X_test"

# load the single final model from 06_model_training.ipynb
with open("../artifacts/models/rf_final_model.pkl", "rb") as f:
    rf_final_model = pickle.load(f)

with open("../artifacts/models/rf_final_model_metadata.json") as f:
    model_metadata = json.load(f)
FINAL_THRESHOLD = model_metadata["final_threshold"]

# predictions regenerated fresh from final_model at the same threshold used in
# training, instead of loading a stale fair_prediction.npy that may belong to a
# different model version.
rf_probs = rf_final_model.predict_proba(scaled_raw_data_X_test)[:, 1]
rf_preds = (rf_probs >= FINAL_THRESHOLD).astype(int)


c:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


#### rebuild real categorical columns from one-hot

In [3]:
# convert one-hot encoded age group columns back into a single categorical column
def onehot_to_category(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "_")]
    return df[cols].idxmax(axis=1).str.replace(prefix + "_", "", regex=False)

sensitive_df_train = pd.DataFrame({
    "age_group": onehot_to_category(scaled_raw_data_X_train, "age_group")
})

sensitive_df_test = pd.DataFrame({
    "age_group": onehot_to_category(scaled_raw_data_X_test, "age_group")
})


Demographic Parity Difference (DPD) & Equalized Odds Difference (EOD)
> - **What it is:** **DPD** = the biggest gap, across sensitive groups (e.g. age bands), in the rate at which the model predicts "default". **EOD** = the biggest gap in error rates (false positive / false negative rate) between groups. Both are 0 in a perfectly fair model, and computed here per-group via `MetricFrame`.
> - **Why we use it:** A model can look accurate overall but still make worse errors for one age group than another — DPD/EOD are the standard fairness metrics for catching exactly that, which matters for a lending model that must not discriminate.


## 2. Fairness metric function (industry-standard `MetricFrame`)

**Industry rule:** always report both a single summary number (demographic parity / equalized odds difference) *and* the full per-group breakdown. A summary number can hide a problem that only shows up in one subgroup.

In [4]:
# --- fairness check function (multi-attribute) ---
def check_fairness_on_test(y_test, y_pred, sensitive_attr):
    """Check fairness metrics on unseen test data using MetricFrame."""
    y_test_clean = np.asarray(y_test).ravel()
    y_pred_clean = np.asarray(y_pred).ravel()
    sensitive_clean = np.asarray(sensitive_attr).ravel()

    dpd = demographic_parity_difference(y_test_clean, y_pred_clean, sensitive_features=sensitive_clean)
    eod = equalized_odds_difference(y_test_clean, y_pred_clean, sensitive_features=sensitive_clean)

    print("=== GLOBAL FAIRNESS METRICS ===")
    print(f"Demographic Parity Difference : {dpd:.4f} (ideal: 0.0000)")
    print(f"Equalized Odds Difference     : {eod:.4f} (ideal: 0.0000)")

    metrics_dict = {
        'predicted_default_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate,
        'accuracy': accuracy_score,
    }
    metric_frame = MetricFrame(
        metrics=metrics_dict, y_true=y_test_clean, y_pred=y_pred_clean, sensitive_features=sensitive_clean
    )
    print("\n=== GROUP-WISE METRICS ===")
    print(metric_frame.by_group)

    return dpd, eod, metric_frame.by_group


## 3. Baseline fairness check (before any correction)

**Industry rule:** pick the sensitive attribute up front based on what you actually need to be fair about (e.g. `age_group` for age-discrimination compliance), and use that *same* attribute for every step below. Switching sensitive attributes partway through an analysis makes before/after comparisons meaningless.

In [5]:
# run fairness check on the baseline model's predictions using age group as the sensitive attribute
SENSITIVE_ATTR_COLUMN = "age_group_18-25"  # locked for the whole notebook — do not change mid-analysis

dpd_score, eod_score, group_metrics_df = check_fairness_on_test(
    y_test=y_test,
    y_pred=rf_preds,
    sensitive_attr=sensitive_df_test['age_group'],
)


=== GLOBAL FAIRNESS METRICS ===
Demographic Parity Difference : 0.2691 (ideal: 0.0000)
Equalized Odds Difference     : 0.2979 (ideal: 0.0000)

=== GROUP-WISE METRICS ===
                     predicted_default_rate  false_positive_rate  \
sensitive_feature_0                                                
18-25                              0.240832             0.065719   
26-35                              0.203988             0.059279   
36-45                              0.236620             0.089606   
46-60                              0.201439             0.063063   
60+                                0.470588             0.357143   

                     false_negative_rate  accuracy  
sensitive_feature_0                                 
18-25                           0.171633  0.909962  
26-35                           0.245950  0.901840  
36-45                           0.223684  0.881690  
46-60                           0.250000  0.899281  
60+                             0.0

## 4. Decision: is correction needed?

**Industry rule:** define the "acceptable" threshold for fairness differences *before* looking at the number (e.g. "we will correct if |difference| > 0.10"), so the decision isn't reverse-engineered to fit what's convenient.

Common industry threshold: differences below ~0.10 are usually considered acceptable; above that, correction is applied.

In [6]:
FAIRNESS_ACCEPTABLE_THRESHOLD = 0.10

needs_correction = abs(eod_score) > FAIRNESS_ACCEPTABLE_THRESHOLD
print(f"Equalized Odds Difference: {eod_score:.4f}")
print(f"Correction needed? {needs_correction} (threshold = {FAIRNESS_ACCEPTABLE_THRESHOLD})")


Equalized Odds Difference: 0.2979
Correction needed? True (threshold = 0.1)


> ThresholdOptimizer (post-processing fairness correction)
> - **What it is:** A technique that keeps the trained model unchanged but picks a *different decision threshold per sensitive group* so that the chosen fairness constraint (here `equalized_odds`) is satisfied as closely as possible.
> - **Why we use it (and only this method):** It's applied after training, so it doesn't require retraining or touching the model that's already been validated everywhere else in the pipeline — simpler to audit than in-training fairness methods.


## 5. Fairness correction (`ThresholdOptimizer`, `equalized_odds`)


In [7]:
def apply_fairness_correction(model, needs_correction, X_train, y_train, X_test,
                               sensitive_train, sensitive_test, baseline_preds,
                               constraints="equalized_odds", objective="accuracy_score"):
    """
    Applies ThresholdOptimizer-based fairness correction if needed.
    Returns fair_preds and the fitted optimizer (or None if correction skipped).
    """

    if needs_correction:
        optimizer = ThresholdOptimizer(
            estimator=model,
            constraints=constraints,
            predict_method="predict_proba",
            objective=objective,
            prefit=True,
        )

        optimizer.fit(
            X_train, y_train,
            sensitive_features=sensitive_train,
        )
        fair_preds = optimizer.predict(
            X_test,
            sensitive_features=sensitive_test,
        )
    else:
        # No correction needed — fair_preds is just the original model's predictions.
        fair_preds = baseline_preds
        optimizer = None
        print("Skipping correction — baseline fairness already within acceptable range.")

    return fair_preds, optimizer

In [8]:
# apply fairness correction to the baseline model
fair_preds, optimizer = apply_fairness_correction(
    model=rf_final_model,
    needs_correction=needs_correction,
    X_train=raw_X_train_prep,
    y_train=y_train,
    X_test=raw_X_test_prep,
    sensitive_train=sensitive_df_train['age_group'],
    sensitive_test=sensitive_df_test['age_group'],
    baseline_preds=rf_preds,
)

## 6. Before vs. after comparison (both computed on the SAME model lineage now)

In [9]:
# function to compare fairness and accuracy metrics before and after correction
def compare_fairness_before_after(y_test, baseline_preds, fair_preds, sensitive_features, check_fairness_fn=check_fairness_on_test):
    """Prints and returns fairness + accuracy comparison before/after correction."""

    print("----- BEFORE correction -----")
    dpd_before, eod_before, _ = check_fairness_fn(y_test, baseline_preds, sensitive_features)

    print("\n----- AFTER correction -----")
    dpd_after, eod_after, group_metrics_after = check_fairness_fn(y_test, fair_preds, sensitive_features)

    acc_before = accuracy_score(y_test, baseline_preds)
    acc_after = accuracy_score(y_test, fair_preds)

    print(f"\nAccuracy before: {acc_before:.4f}")
    print(f"Accuracy after:  {acc_after:.4f}")
    print(classification_report(y_test, fair_preds))

    return {
        "dpd_before": dpd_before,
        "eod_before": eod_before,
        "dpd_after": dpd_after,
        "eod_after": eod_after,
        "group_metrics_after": group_metrics_after,
        "acc_before": acc_before,
        "acc_after": acc_after,
    }

In [10]:
# compare fairness metrics before and after correction
results = compare_fairness_before_after(
    y_test=y_test,
    baseline_preds=rf_preds,
    fair_preds=fair_preds,
    sensitive_features=sensitive_df_test["age_group"],
)


----- BEFORE correction -----
=== GLOBAL FAIRNESS METRICS ===
Demographic Parity Difference : 0.2691 (ideal: 0.0000)
Equalized Odds Difference     : 0.2979 (ideal: 0.0000)

=== GROUP-WISE METRICS ===
                     predicted_default_rate  false_positive_rate  \
sensitive_feature_0                                                
18-25                              0.240832             0.065719   
26-35                              0.203988             0.059279   
36-45                              0.236620             0.089606   
46-60                              0.201439             0.063063   
60+                                0.470588             0.357143   

                     false_negative_rate  accuracy  
sensitive_feature_0                                 
18-25                           0.171633  0.909962  
26-35                           0.245950  0.901840  
36-45                           0.223684  0.881690  
46-60                           0.250000  0.899281  
60+  

EO is too high, probably there is low range of group distort the TPR and FPR 

In [11]:
# check
print(raw_X_train['age_group'].value_counts())
print(raw_X_test['age_group'].value_counts())

age_group
18-25    8578
26-35    7707
36-45    1505
46-60     325
60+        37
Name: count, dtype: int64
age_group
18-25    3654
26-35    3260
36-45     710
46-60     139
60+        17
Name: count, dtype: int64


In [12]:
# merge the small 60+ age group into the 46-60 group
sensitive_df_train['age_group'] = sensitive_df_train['age_group'].replace('60+', '46-60')
sensitive_df_test['age_group'] = sensitive_df_test['age_group'].replace('60+', '46-60')

In [13]:
# re-apply fairness correction after merging the small age group
fair_preds, optimizer = apply_fairness_correction(
    model=rf_final_model,
    needs_correction=needs_correction,
    X_train=raw_X_train_prep,
    y_train=y_train,
    X_test=raw_X_test_prep,
    sensitive_train=sensitive_df_train['age_group'],
    sensitive_test=sensitive_df_test['age_group'],
    baseline_preds=rf_preds,
)

In [14]:
# re-compare fairness metrics after merging the small age group
results = compare_fairness_before_after(
    y_test=y_test,
    baseline_preds=rf_preds,
    fair_preds=fair_preds,
    sensitive_features=sensitive_df_test["age_group"],
)

----- BEFORE correction -----
=== GLOBAL FAIRNESS METRICS ===
Demographic Parity Difference : 0.0368 (ideal: 0.0000)
Equalized Odds Difference     : 0.0743 (ideal: 0.0000)

=== GROUP-WISE METRICS ===
                     predicted_default_rate  false_positive_rate  \
sensitive_feature_0                                                
18-25                              0.240832             0.065719   
26-35                              0.203988             0.059279   
36-45                              0.236620             0.089606   
46-60                              0.230769             0.096000   

                     false_negative_rate  accuracy  
sensitive_feature_0                                 
18-25                           0.171633  0.909962  
26-35                           0.245950  0.901840  
36-45                           0.223684  0.881690  
46-60                           0.225806  0.878205  

----- AFTER correction -----
=== GLOBAL FAIRNESS METRICS ===
Demographic

## 7. Save the fairness decision (industry rule: audit trail, not just a printout)

A fairness correction that isn't recorded anywhere is not auditable later — regulators, internal risk teams, or future engineers need to know what was checked, what was decided, and why.

In [15]:
# save the fairness report and optimizer (if correction was applied)
fairness_record = {
    "sensitive_attribute": SENSITIVE_ATTR_COLUMN,
    "acceptable_threshold": FAIRNESS_ACCEPTABLE_THRESHOLD,
    "correction_applied": bool(needs_correction),
    "method": "ThresholdOptimizer (equalized_odds)" if needs_correction else "none",
    "metrics_before": {"demographic_parity_diff": round(float(results['dpd_before']), 4), "equalized_odds_diff": round(float(results['eod_before']), 4)},
    "metrics_after": {"demographic_parity_diff": round(float(results['dpd_after']), 4), "equalized_odds_diff": round(float(results['eod_after']), 4)},
}

with open("../artifacts/models/fairness_report.json", "w") as f:
    json.dump(fairness_record, f, indent=2)

if needs_correction:
    with open("../artifacts/models/fairness_optimizer.pkl", "wb") as f:
        pickle.dump(optimizer, f)

print(json.dumps(fairness_record, indent=2))


{
  "sensitive_attribute": "age_group_18-25",
  "acceptable_threshold": 0.1,
  "correction_applied": true,
  "method": "ThresholdOptimizer (equalized_odds)",
  "metrics_before": {
    "demographic_parity_diff": 0.0368,
    "equalized_odds_diff": 0.0743
  },
  "metrics_after": {
    "demographic_parity_diff": 0.0558,
    "equalized_odds_diff": 0.1164
  }
}
